In [1]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction import FeatureHasher
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
from joblib import dump, load
import warnings
from category_encoders import TargetEncoder  # Added import
import matplotlib.pyplot as plt
import os

In [2]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

class ChampionHasher(BaseEstimator, TransformerMixin):
    """Feature hashing for champion compositions"""
    def __init__(self, n_features=300):
        self.n_features = n_features
        self.hasher = FeatureHasher(n_features=n_features, input_type='string')
        self.seen_champions = set()

    def fit(self, X, y=None):
        for _, row in X.iterrows():
            self._process_team(row['blue_picks'])
            self._process_team(row['red_picks'])
        return self

    def _process_team(self, picks):
        for champ in picks:
            self.seen_champions.add(champ.strip().title())

    def transform(self, X):
        def hash_team(picks):
            return self.hasher.transform([picks]).toarray()[0]

        blue_features = X['blue_picks'].apply(hash_team)
        red_features = X['red_picks'].apply(hash_team)
        return np.hstack([np.vstack(blue_features), np.vstack(red_features)])

class TeamEncoder(BaseEstimator, TransformerMixin):
    """Encodes team historical performance"""
    def __init__(self):
        self.encoder = TargetEncoder(smoothing=20)

    def fit(self, X, y):
        # Create team-target pairs for both sides
        teams = pd.concat([
            X['blue_team'].rename('team'),
            X['red_team'].rename('team')
        ])

        # Duplicate target values for both teams in each match
        y_duplicated = pd.concat([y, y])

        self.encoder.fit(teams.values.reshape(-1, 1), y_duplicated)
        return self

    def transform(self, X):
        blue_enc = self.encoder.transform(X['blue_team'].values.reshape(-1, 1))
        red_enc = self.encoder.transform(X['red_team'].values.reshape(-1, 1))
        return np.hstack([blue_enc, red_enc])

class GameLengthPredictor:
    def __init__(self):
        self.pipeline = None
        self.champion_hasher = None
        self.team_encoder = None

    def load_data(self, path):
        dtype = {
            'gameid': 'string',
            'side': 'string',
            'champion': 'string',
            'teamname': 'string',
            'gamelength': 'int32',
            'participantid': 'string'
        }

        df = pd.read_csv(
            path,
            usecols=['gameid', 'side', 'champion', 'teamname', 'gamelength', 'participantid'],
            dtype=dtype,
            na_filter=False
        )

        matches = []
        for game_id in df['gameid'].unique():
            game_data = df[df['gameid'] == game_id]
            game_data = game_data[~game_data['participantid'].isin(['100', '200'])]  # Remove team rows

            if len(game_data) != 10:
                continue

            try:
                blue_team = game_data[game_data['side'] == 'Blue']
                red_team = game_data[game_data['side'] == 'Red']

                matches.append({
                    'blue_picks': blue_team['champion'].str.strip().str.title().tolist(),
                    'red_picks': red_team['champion'].str.strip().str.title().tolist(),
                    'blue_team': blue_team['teamname'].iloc[0],
                    'red_team': red_team['teamname'].iloc[0],
                    'gamelength': game_data['gamelength'].iloc[0]  # In seconds
                })

            except Exception as e:
                continue

        return pd.DataFrame(matches)

    def train(self, data_path, save_path='game_length_model.joblib'):
        df = self.load_data(data_path)
        # df = pd.concat([df, self.load_data("matches2024.csv")], ignore_index=True, sort=False)

        if len(df) < 50:
            print(f"Need at least 50 matches, got {len(df)}")
            return

        X = df[['blue_picks', 'red_picks', 'blue_team', 'red_team']]
        y = df['gamelength'] / 60  # Convert to minutes

        # In the pipeline definition:
        self.pipeline = Pipeline([
            ('features', ColumnTransformer([
                ('champions', ChampionHasher(), ['blue_picks', 'red_picks']),
                ('teams', TeamEncoder(), ['blue_team', 'red_team'])
            ])),
            ('regressor', HistGradientBoostingRegressor(
                max_iter=2000,
                learning_rate=0.05,
                max_depth=10,
                early_stopping=True,
                random_state=42
            ))
        ])

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        self.pipeline.fit(X_train, y_train)

        # Evaluate model
        train_pred = self.pipeline.predict(X_train)
        test_pred = self.pipeline.predict(X_test)

        # Compute standard deviation of residuals (for confidence intervals)
        self.train_residual_std = np.std(y_train - train_pred)

        print(f"Training RMSE: {np.sqrt(mean_squared_error(y_train, train_pred)):.2f} mins")
        print(f"Validation RMSE: {np.sqrt(mean_squared_error(y_test, test_pred)):.2f} mins")
        print(f"Training MAE: {mean_absolute_error(y_train, train_pred):.2f} mins")
        print(f"Validation MAE: {mean_absolute_error(y_test, test_pred):.2f} mins")

        # Save components
        self.champion_hasher = self.pipeline.named_steps['features'].transformers_[0][1]
        self.team_encoder = self.pipeline.named_steps['features'].transformers_[1][1]
        dump(self.pipeline, save_path)

    def load(self, model_path='game_length_model.joblib'):
        self.pipeline = load(model_path)
        self.champion_hasher = self.pipeline.named_steps['features'].transformers_[0][1]
        self.team_encoder = self.pipeline.named_steps['features'].transformers_[1][1]
        return self

    def predict_duration(self, blue_picks, red_picks, blue_team, red_team):
        input_df = pd.DataFrame([{
            'blue_picks': [c.strip().title() for c in blue_picks],
            'red_picks': [c.strip().title() for c in red_picks],
            'blue_team': blue_team,
            'red_team': red_team
        }])

        try:
            pred = self.pipeline.predict(input_df)[0]
            margin = 2 * self.train_residual_std

            return {
                'predicted_minutes': round(pred, 1),
                'confidence_interval': (
                    round(max(0, pred - margin), 1),
                    round(pred + margin, 1)
                )
            }
        except Exception as e:
            return {'error': str(e)}

In [4]:
predictor = GameLengthPredictor()
predictor.train('matches.csv')

# Sample prediction
result = predictor.predict_duration(
    blue_picks=["Darius", "Nidalee", "Jayce", "Ezreal", "Alistar"],
    red_picks=["Ambessa", "Skarner", "Aurora", "Varus", "Rakan"],
    blue_team="BoostGate Esports",
    red_team="BBL Dark Passage"
)

if 'error' in result:
    print(f"Error: {result['error']}")
else:
    print(f"Predicted Game Length: {result['predicted_minutes']} minutes")
    print(f"95% Confidence Range: {result['confidence_interval'][0]} - {result['confidence_interval'][1]} minutes")

Training RMSE: 4.83 mins
Validation RMSE: 5.49 mins
Training MAE: 3.77 mins
Validation MAE: 4.27 mins
Predicted Game Length: 33.4 minutes
95% Confidence Range: 23.7 - 43.1 minutes
